# About: Docker Swarmの準備

構築済みのVM上に、DockerEngineをインストールし、Docker Swarmを準備する。

# 設定情報

このNotebookで行う設定は、以下のようにする。

- Docker関連のディレクトリ （コンテナデータ、イメージ、一時ファイル用ディレクトリ等）
  - NIIのベアメタルマシンを想定する場合 ... Docker関係のディレクトリは `/mnt`に配置する
  - それ以外 ... ストレージサイズ等を考慮して、適切なパスを設定すること
- プライベートレジストリ
  - 公開のレジストリのみ使用する場合 ... なし
  - クラウド運用チームで利用する場合 ... Dockerのプライベートレジストリを使用するので、プライベートレジストリのホスト情報を明示する

docker_optsの定義方法は[DAEMON CONFIGURATION FILE](https://docs.docker.com/engine/reference/commandline/dockerd/#daemon-configuration-file)を参照。

In [1]:
docker_tmp = '/var/lib/docker/tmp'
docker_base = '/var/lib/docker'
docker_opts ={"data-root": docker_base, "insecure-registries": []}

# Notebookと環境のBinding

Inventory中のgroup名でBind対象ホスト(Docker Engineをインストールしたいホスト)を指示する。

Ansible Inventory中に定義されているグループ( `[グループ名]` のように`[]`で囲まれた定義 )のうち、CoursewareHub構築に使うグループ名を指定します。

In [2]:
hosts_file = './hosts'
target_group = 'cwhtest0001'

# 接続確認
%env ANSIBLE_INVENTORY={hosts_file}
!ansible -m ping {target_group}

env: ANSIBLE_INVENTORY=./hosts
10.1.1.144 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.157 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.236 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


作業ディレクトリを作成

In [3]:
import tempfile
temp_dir = tempfile.mkdtemp()
temp_dir

'/tmp/tmpbee44q4h'

# Binding対象の確認

[Prerequisites](https://docs.docker.com/engine/installation/linux/ubuntulinux/#prerequisites)に示されているとおり、このNotebookを使ってDockerをインストールする対象のホストは、以下の条件を満たしている必要がある。

もし、このインストール手順に失敗したら、**この条件を満たす状態にマシンを戻す(パッケージの削除, マシンの再プロビジョニングなど)**ことで、**(このNotebookによって)Dockerをインストール可能な状態に戻す**ことができる。

## 64bit版を使う

Docker Engineを動作させるには64bit版が必要。

In [4]:
!ansible -a 'uname -m' {target_group}

10.1.1.144 | CHANGED | rc=0 >>
x86_64
10.1.1.236 | CHANGED | rc=0 >>
x86_64
10.1.1.157 | CHANGED | rc=0 >>
x86_64


## kernel versionは最低3.10

3.10未満の古いバージョンの場合はDockerの機能の一部が使えなかったり、データロストやpanicを生じる可能性がある。

そのため、以下のバージョン表示が3.10以上であることを確認しておく。

In [5]:
!ansible -a 'uname -r' {target_group}

10.1.1.144 | CHANGED | rc=0 >>
3.10.0-1062.12.1.el7.x86_64
10.1.1.236 | CHANGED | rc=0 >>
3.10.0-1062.12.1.el7.x86_64
10.1.1.157 | CHANGED | rc=0 >>
3.10.0-1062.12.1.el7.x86_64


## CentOSは7を想定



In [6]:
!ansible -b -m yum -a 'name=redhat-lsb' {target_group}

10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "redhat-lsb"
        ]
    },
    "msg": "",
    "rc": 0,
    "results": [
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package redhat-lsb.x86_64 0:4.1-27.el7.centos.1 will be installed\n--> Processing Dependency: redhat-lsb-printing = 4.1-27.el7.centos.1 for package: redhat-lsb-4.1-27.el7.centos.1.x86_64\n--> Processing Dependency: redhat-lsb-languages = 4.1-27.el7.centos.1 for package: redhat-lsb-4.1-27.el7.centos.1.x86_64\n--> Processing Dependency: redhat-lsb-desktop(x86-64) = 4.1-27.el7.centos.1 for package: redhat-lsb-4.1-27.el7.centos.1.x86_64\n--> Processing Dependency: redhat-lsb-cxx(x86-64) =

10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "redhat-lsb"
        ]
    },
    "msg": "warning: /var/cache/yum/x86_64/7/base/packages/adobe-mappings-cmap-deprecated-20171205-3.el7.noarch.rpm: Header V3 RSA/SHA256 Signature, key ID f4a80eb5: NOKEY\nImporting GPG key 0xF4A80EB5:\n Userid     : \"CentOS-7 Key (CentOS 7 Official Signing Key) <security@centos.org>\"\n Fingerprint: 6341 ab27 53d7 8a78 a7c2 7bb1 24c6 a8a7 f4a8 0eb5\n Package    : centos-release-7-7.1908.0.el7.centos.x86_64 (installed)\n From       : /etc/pki/rpm-gpg/RPM-GPG-KEY-CentOS-7\n",
    "rc": 0,
    "results": [
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package redhat-lsb.x86

10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "redhat-lsb"
        ]
    },
    "msg": "warning: /var/cache/yum/x86_64/7/base/packages/adobe-mappings-cmap-deprecated-20171205-3.el7.noarch.rpm: Header V3 RSA/SHA256 Signature, key ID f4a80eb5: NOKEY\nImporting GPG key 0xF4A80EB5:\n Userid     : \"CentOS-7 Key (CentOS 7 Official Signing Key) <security@centos.org>\"\n Fingerprint: 6341 ab27 53d7 8a78 a7c2 7bb1 24c6 a8a7 f4a8 0eb5\n Package    : centos-release-7-7.1908.0.el7.centos.x86_64 (installed)\n From       : /etc/pki/rpm-gpg/RPM-GPG-KEY-CentOS-7\n",
    "rc": 0,
    "results": [
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package redhat-lsb.x86

In [7]:
!ansible -a 'lsb_release -a' {target_group}

10.1.1.144 | CHANGED | rc=0 >>
LSB Version:	:core-4.1-amd64:core-4.1-noarch:cxx-4.1-amd64:cxx-4.1-noarch:desktop-4.1-amd64:desktop-4.1-noarch:languages-4.1-amd64:languages-4.1-noarch:printing-4.1-amd64:printing-4.1-noarch
Distributor ID:	CentOS
Description:	CentOS Linux release 7.7.1908 (Core)
Release:	7.7.1908
Codename:	Core
10.1.1.236 | CHANGED | rc=0 >>
LSB Version:	:core-4.1-amd64:core-4.1-noarch:cxx-4.1-amd64:cxx-4.1-noarch:desktop-4.1-amd64:desktop-4.1-noarch:languages-4.1-amd64:languages-4.1-noarch:printing-4.1-amd64:printing-4.1-noarch
Distributor ID:	CentOS
Description:	CentOS Linux release 7.7.1908 (Core)
Release:	7.7.1908
Codename:	Core
10.1.1.157 | CHANGED | rc=0 >>
LSB Version:	:core-4.1-amd64:core-4.1-noarch:cxx-4.1-amd64:cxx-4.1-noarch:desktop-4.1-amd64:desktop-4.1-noarch:languages-4.1-amd64:languages-4.1-noarch:printing-4.1-amd64:printing-4.1-noarch
Distributor ID:	CentOS
Description:	CentOS Linux release 7.7.1908 (Core)
Release:	7.7.1908
Codename:	Core


## 古いパッケージの削除

古いパッケージの削除

In [8]:
!ansible -b -m yum -a 'name=docker,docker-client,docker-client-latest,docker-common,docker-latest,docker-latest-logrotate,docker-logrotate,docker-selinux,docker-engine-selinux,docker-engine state=absent' {target_group}

10.1.1.236 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "msg": "",
    "rc": 0,
    "results": [
        "docker is not installed",
        "docker-client is not installed",
        "docker-client-latest is not installed",
        "docker-common is not installed",
        "docker-latest is not installed",
        "docker-latest-logrotate is not installed",
        "docker-logrotate is not installed",
        "docker-selinux is not installed",
        "docker-engine-selinux is not installed",
        "docker-engine is not installed"
    ]
}
10.1.1.144 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "msg": "",
    "rc": 0,
    "results": [
        "docker is not installed",
        "docker-client is not installed",
        "docker-client-latest is not installed",
        "docker-common is not installed",
        "docker-lat

# Docker Engineのインストール

Bind対象にDocker Engineをインストールする。

## リポジトリの設定

In [9]:
!ansible -b -m yum -a 'name=yum-utils,device-mapper-persistent-data,lvm2' {target_group}

10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "device-mapper-persistent-data",
            "lvm2"
        ]
    },
    "msg": "",
    "rc": 0,
    "results": [
        "yum-utils-1.1.31-52.el7.noarch providing yum-utils is already installed",
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package device-mapper-persistent-data.x86_64 0:0.8.5-3.el7_9.2 will be installed\n--> Processing Dependency: libaio.so.1(LIBAIO_0.4)(64bit) for package: device-mapper-persistent-data-0.8.5-3.el7_9.2.x86_64\n--> Processing Dependency: libaio.so.1(LIBAIO_0.1)(64bit) for package: device-mapper-persistent-data-0.8.5-3.el7_9.2.x86_64\n--> Processing Dependency: libaio.s

10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "device-mapper-persistent-data",
            "lvm2"
        ]
    },
    "msg": "",
    "rc": 0,
    "results": [
        "yum-utils-1.1.31-52.el7.noarch providing yum-utils is already installed",
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package device-mapper-persistent-data.x86_64 0:0.8.5-3.el7_9.2 will be installed\n--> Processing Dependency: libaio.so.1(LIBAIO_0.4)(64bit) for package: device-mapper-persistent-data-0.8.5-3.el7_9.2.x86_64\n--> Processing Dependency: libaio.so.1(LIBAIO_0.1)(64bit) for package: device-mapper-persistent-data-0.8.5-3.el7_9.2.x86_64\n--> Processing Dependency: libaio.s

In [10]:
!ansible -b -a 'yum-config-manager --add-repo https://download.docker.com/linux/centos/docker-ce.repo' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
Loaded plugins: fastestmirror
adding repo from: https://download.docker.com/linux/centos/docker-ce.repo
grabbing file https://download.docker.com/linux/centos/docker-ce.repo to /etc/yum.repos.d/docker-ce.repo
repo saved to /etc/yum.repos.d/docker-ce.repo
10.1.1.144 | CHANGED | rc=0 >>
Loaded plugins: fastestmirror
adding repo from: https://download.docker.com/linux/centos/docker-ce.repo
grabbing file https://download.docker.com/linux/centos/docker-ce.repo to /etc/yum.repos.d/docker-ce.repo
repo saved to /etc/yum.repos.d/docker-ce.repo
10.1.1.236 | CHANGED | rc=0 >>
Loaded plugins: fastestmirror
adding repo from: https://download.docker.com/linux/centos/docker-ce.repo
grabbing file https://download.docker.com/linux/centos/docker-ce.repo to /etc/yum.repos.d/docker-ce.repo
repo saved to /etc/yum.repos.d/docker-ce.repo


## パッケージのインストール

`docker-ce` パッケージをインストールする。

In [11]:
!ansible -b -m yum -a 'name=docker-ce' {target_group}

10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "docker-ce"
        ]
    },
    "msg": "warning: /var/cache/yum/x86_64/7/docker-ce-stable/packages/docker-ce-20.10.8-3.el7.x86_64.rpm: Header V4 RSA/SHA512 Signature, key ID 621e9f35: NOKEY\nImporting GPG key 0x621E9F35:\n Userid     : \"Docker Release (CE rpm) <docker@docker.com>\"\n Fingerprint: 060a 61c5 1b55 8a7f 742b 77aa c52f eb6b 621e 9f35\n From       : https://download.docker.com/linux/centos/gpg\n",
    "rc": 0,
    "results": [
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package docker-ce.x86_64 3:20.10.8-3.el7 will be installed\n--> Processing Dependency: container-selinux >= 2:2.74 for p

10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "changes": {
        "installed": [
            "docker-ce"
        ]
    },
    "msg": "warning: /var/cache/yum/x86_64/7/docker-ce-stable/packages/containerd.io-1.4.9-3.1.el7.x86_64.rpm: Header V4 RSA/SHA512 Signature, key ID 621e9f35: NOKEY\nImporting GPG key 0x621E9F35:\n Userid     : \"Docker Release (CE rpm) <docker@docker.com>\"\n Fingerprint: 060a 61c5 1b55 8a7f 742b 77aa c52f eb6b 621e 9f35\n From       : https://download.docker.com/linux/centos/gpg\n",
    "rc": 0,
    "results": [
        "Loaded plugins: fastestmirror\nLoading mirror speeds from cached hostfile\n * base: download.cf.centos.org\n * extras: download.cf.centos.org\n * updates: download.cf.centos.org\nResolving Dependencies\n--> Running transaction check\n---> Package docker-ce.x86_64 3:20.10.8-3.el7 will be installed\n--> Processing Dependency: container-selinux >= 2:2.74 f

## Docker Engineの設定変更

あらかじめ定義した設定情報にしたがい、Docker Engineに与えるDefault Configを指定する。

In [12]:
import os
import json
with open(os.path.join(temp_dir, 'daemon.json'), 'w') as f:
    f.write(json.dumps(docker_opts))
!cat {temp_dir}/daemon.json

{"data-root": "/var/lib/docker", "insecure-registries": []}

In [13]:
import os
with open(os.path.join(temp_dir, 'tmpdir.conf'), 'w') as f:
    f.write('''# Systemd drop-in configuration for Docker
[Service]
Environment="DOCKER_TMPDIR={docker_tmp}"'''.format(docker_tmp=docker_tmp))
!cat {temp_dir}/tmpdir.conf

# Systemd drop-in configuration for Docker
[Service]
Environment="DOCKER_TMPDIR=/var/lib/docker/tmp"

ローカルに作った configファイル を、Bind対象の/etc/default/dockerにコピーし、Docker Engineに反映する。

In [14]:
!ansible -b -m file -a 'path=/etc/docker state=directory' {target_group}
!ansible -b -m copy -a 'src={temp_dir}/daemon.json dest=/etc/docker/daemon.json' {target_group}
!ansible -b -m file -a 'path=/etc/systemd/system/docker.service.d state=directory' {target_group}
!ansible -b -m copy -a 'src={temp_dir}/tmpdir.conf dest=/etc/systemd/system/docker.service.d/tmpdir.conf' {target_group}

!ansible -b -m file -a 'path={docker_tmp} state=directory' {target_group}
!ansible -b -a 'systemctl daemon-reload' {target_group}
!ansible -b -m service -a 'name=docker state=restarted enabled=yes' {target_group}

10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "gid": 0,
    "group": "root",
    "mode": "0755",
    "owner": "root",
    "path": "/etc/docker",
    "secontext": "unconfined_u:object_r:etc_t:s0",
    "size": 6,
    "state": "directory",
    "uid": 0
}
10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "gid": 0,
    "group": "root",
    "mode": "0755",
    "owner": "root",
    "path": "/etc/docker",
    "secontext": "unconfined_u:object_r:etc_t:s0",
    "size": 6,
    "state": "directory",
    "uid": 0
}
10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "gid": 0,
    "group": "root",
    "mode": "0755",
    "owner": "root",
    "path": "/etc/docker",
    "secontext": "unconfined_u:object_r:etc_t:s0",
    "size": 6,
   

10.1.1.236 | CHANGED | rc=0 >>

10.1.1.157 | CHANGED | rc=0 >>

10.1.1.144 | CHANGED | rc=0 >>

10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "enabled": true,
    "name": "docker",
    "state": "started",
    "status": {
        "ActiveEnterTimestampMonotonic": "0",
        "ActiveExitTimestampMonotonic": "0",
        "ActiveState": "inactive",
        "After": "network-online.target basic.target firewalld.service system.slice docker.socket systemd-journald.socket containerd.service",
        "AllowIsolate": "no",
        "AmbientCapabilities": "0",
        "AssertResult": "no",
        "AssertTimestampMonotonic": "0",
        "Before": "shutdown.target",
        "BlockIOAccounting": "no",
        "BlockIOWeight": "18446744073709551615",
        "CPUAccounting": "no",
        "CPUQuotaPerSecUSec": "infinity",
        "CPUSchedulingPolicy": "0",
        "CPUSchedulingPriority": "0",
        "CP

10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "enabled": true,
    "name": "docker",
    "state": "started",
    "status": {
        "ActiveEnterTimestampMonotonic": "0",
        "ActiveExitTimestampMonotonic": "0",
        "ActiveState": "inactive",
        "After": "firewalld.service docker.socket network-online.target containerd.service systemd-journald.socket basic.target system.slice",
        "AllowIsolate": "no",
        "AmbientCapabilities": "0",
        "AssertResult": "no",
        "AssertTimestampMonotonic": "0",
        "Before": "shutdown.target",
        "BlockIOAccounting": "no",
        "BlockIOWeight": "18446744073709551615",
        "CPUAccounting": "no",
        "CPUQuotaPerSecUSec": "infinity",
        "CPUSchedulingPolicy": "0",
        "CPUSchedulingPriority": "0",
        "CPUSchedulingResetOnFork": "no",
        "CPUShares": "18446744073709551615",
        "CanIsolate"

念のため、Docker Engineにより /mnt/docker, /mnt/docker-tmp (変更していればそのディレクトリ) にファイルが作成されていることを確認する。

In [15]:
!ansible -b -a 'ls -la {docker_tmp} {docker_base}' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
/var/lib/docker:
total 4
drwx--x--x. 13 root root  167 Sep 13 09:08 .
drwxr-xr-x. 29 root root 4096 Sep 13 09:08 ..
drwx--x--x.  4 root root  120 Sep 13 09:08 buildkit
drwx-----x.  2 root root    6 Sep 13 09:08 containers
drwx------.  3 root root   22 Sep 13 09:08 image
drwxr-x---.  3 root root   19 Sep 13 09:08 network
drwx-----x.  3 root root   40 Sep 13 09:08 overlay2
drwx------.  4 root root   32 Sep 13 09:08 plugins
drwx------.  2 root root    6 Sep 13 09:08 runtimes
drwx------.  2 root root    6 Sep 13 09:08 swarm
drwx------.  2 root root    6 Sep 13 09:08 tmp
drwx------.  2 root root    6 Sep 13 09:08 trust
drwx-----x.  2 root root   50 Sep 13 09:08 volumes

/var/lib/docker/tmp:
total 0
drwx------.  2 root root   6 Sep 13 09:08 .
drwx--x--x. 13 root root 167 Sep 13 09:08 ..
10.1.1.144 | CHANGED | rc=0 >>
/var/lib/docker:
total 4
drwx--x--x. 13 root root  167 Sep 13 09:08 .
drwxr-xr-x. 29 root root 4096 Sep 13 09:08 ..
drwx--x--x.  4 root root  120 

サービスの状態を確認

In [16]:
!ansible -b -a 'systemctl status docker' {target_group}

10.1.1.236 | CHANGED | rc=0 >>
● docker.service - Docker Application Container Engine
   Loaded: loaded (/usr/lib/systemd/system/docker.service; enabled; vendor preset: disabled)
  Drop-In: /etc/systemd/system/docker.service.d
           └─tmpdir.conf
   Active: active (running) since Mon 2021-09-13 09:08:35 UTC; 4s ago
     Docs: https://docs.docker.com
 Main PID: 15577 (dockerd)
    Tasks: 8
   Memory: 30.6M
   CGroup: /system.slice/docker.service
           └─15577 /usr/bin/dockerd -H fd:// --containerd=/run/containerd/containerd.sock

Sep 13 09:08:33 ip-10-1-1-236.ap-northeast-1.compute.internal dockerd[15577]: time="2021-09-13T09:08:33.680559747Z" level=info msg="scheme \"unix\" not registered, fallback to default scheme" module=grpc
Sep 13 09:08:33 ip-10-1-1-236.ap-northeast-1.compute.internal dockerd[15577]: time="2021-09-13T09:08:33.680579033Z" level=info msg="ccResolverWrapper: sending update to cc: {[{unix:///run/containerd/containerd.sock  <nil> 0 <nil>}] <nil> <nil>}" modul

マシンのboot時にdockerサービスが起動するようにしておく。

In [17]:
!ansible -b -a 'systemctl enable docker' {target_group}

10.1.1.144 | CHANGED | rc=0 >>

10.1.1.157 | CHANGED | rc=0 >>

10.1.1.236 | CHANGED | rc=0 >>



Docker Engineのバージョンを確認する。

In [18]:
!ansible -b -a 'docker version' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
Client: Docker Engine - Community
 Version:           20.10.8
 API version:       1.41
 Go version:        go1.16.6
 Git commit:        3967b7d
 Built:             Fri Jul 30 19:55:49 2021
 OS/Arch:           linux/amd64
 Context:           default
 Experimental:      true

Server: Docker Engine - Community
 Engine:
  Version:          20.10.8
  API version:      1.41 (minimum version 1.12)
  Go version:       go1.16.6
  Git commit:       75249d8
  Built:            Fri Jul 30 19:54:13 2021
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          1.4.9
  GitCommit:        e25210fe30a0a703442421b0f60afac609f950a3
 runc:
  Version:          1.0.1
  GitCommit:        v1.0.1-0-g4144b63
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0
10.1.1.144 | CHANGED | rc=0 >>
Client: Docker Engine - Community
 Version:           20.10.8
 API version:       1.41
 Go version:        go1.16.6
 Git commit:        3967b7d


Docker Engineの設定状況も確認しておく。

In [19]:
!ansible -b -a 'docker info' {target_group}

10.1.1.144 | CHANGED | rc=0 >>
Client:
 Context:    default
 Debug Mode: false
 Plugins:
  app: Docker App (Docker Inc., v0.9.1-beta3)
  buildx: Build with BuildKit (Docker Inc., v0.6.1-docker)
  scan: Docker Scan (Docker Inc., v0.8.0)

Server:
 Containers: 0
  Running: 0
  Paused: 0
  Stopped: 0
 Images: 0
 Server Version: 20.10.8
 Storage Driver: overlay2
  Backing Filesystem: xfs
  Supports d_type: true
  Native Overlay Diff: true
  userxattr: false
 Logging Driver: json-file
 Cgroup Driver: cgroupfs
 Cgroup Version: 1
 Plugins:
  Volume: local
  Network: bridge host ipvlan macvlan null overlay
  Log: awslogs fluentd gcplogs gelf journald json-file local logentries splunk syslog
 Swarm: inactive
 Runtimes: io.containerd.runc.v2 io.containerd.runtime.v1.linux runc
 Default Runtime: runc
 Init Binary: docker-init
 containerd version: e25210fe30a0a703442421b0f60afac609f950a3
 runc version: v1.0.1-0-g4144b63
 init version: de40ad0
 Security Options:
  seccomp
   Profile: default
 Kernel

# Docker Composeのインストール

*2021/9/7時点* では、docker-composeのバージョンは`1.29.2`となる。

In [20]:
!ansible -b -m shell \
         -a 'curl -L https://github.com/docker/compose/releases/download/1.29.2/docker-compose-`uname -s`-`uname -m` \
                 > /usr/local/bin/docker-compose' {target_group}

[WARNING]: Consider using the get_url or uri module rather than running 'curl'.
If you need to use command because get_url or uri is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.144 | CHANGED | rc=0 >>
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   633  100   633    0     0   1674      0 --:--:-- --:--:-- --:--:--  1683
100 12.1M  100 12.1M    0     0  12.4M      0 --:--:-- --:--:-- --:--:-- 55.3M
10.1.1.236 | CHANGED | rc=0 >>
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   633  100   633    0     0   1725      0 --:--:-- --:--:-- --:--:--  1724
100 12.1M  100 12.1M    0     0  12.5M      0 --:--:-- --:--:-- --:--:-- 12.5M
10.1.1.157 | CHANGED | rc=0 >>
  % Tot

In [21]:
!ansible -b -a 'chmod +x /usr/local/bin/docker-compose' {target_group}

[WARNING]: Consider using the file module with mode rather than running
'chmod'.  If you need to use command because file is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.144 | CHANGED | rc=0 >>

10.1.1.157 | CHANGED | rc=0 >>

10.1.1.236 | CHANGED | rc=0 >>



In [22]:
!ansible -b -a '/usr/local/bin/docker-compose --version' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
docker-compose version 1.29.2, build 5becea4c
10.1.1.144 | CHANGED | rc=0 >>
docker-compose version 1.29.2, build 5becea4c
10.1.1.236 | CHANGED | rc=0 >>
docker-compose version 1.29.2, build 5becea4c


# Docker Engineの動作確認

まずはお試しで、hello-worldイメージを実行してみる。`Hello from Docker`のようなメッセージが表示されたらOK。

In [23]:
!ansible -b -a 'docker run hello-world' {target_group}

10.1.1.157 | CHANGED | rc=0 >>

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world
b8dfde127a29: Pulling fs layer
b8dfde127a29: Verifying Checksum
b8dfde127

Dockerのhello-worldイメージが実行された。OK。

# Docker Composeの動作確認

Docker Composeが動作することも確認しておく。

まずローカルにdocker-compose.ymlファイルを準備。

In [24]:
!mkdir -p {temp_dir}/hello-compose/

In [25]:
%%writefile {temp_dir}/hello-compose/docker-compose.yml
version: '2'
services:
  test-hello-world:
    image: hello-world

Writing /tmp/tmpbee44q4h/hello-compose/docker-compose.yml


作成したdocker-compose.ymlを、Bind対象ホストにアップロードする。

In [26]:
!ansible -b -m copy -a 'src={temp_dir}/hello-compose dest=~' {target_group}

10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "checksum": "c94b05c15796715e5fc48ef742953d092b91a08d",
    "dest": "/root/hello-compose/docker-compose.yml",
    "gid": 0,
    "group": "root",
    "md5sum": "6c332ed8081a56b548fd18193ad03378",
    "mode": "0644",
    "owner": "root",
    "secontext": "system_u:object_r:admin_home_t:s0",
    "size": 66,
    "src": "/home/centos/.ansible/tmp/ansible-tmp-1631524147.6515226-2654-91237019972136/source",
    "state": "file",
    "uid": 0
}
10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": true,
    "checksum": "c94b05c15796715e5fc48ef742953d092b91a08d",
    "dest": "/root/hello-compose/docker-compose.yml",
    "gid": 0,
    "group": "root",
    "md5sum": "6c332ed8081a56b548fd18193ad03378",
    "mode": "0644",
    "owner": "root",
    "secontext": "system_u:object_r:admin_ho

実行してみる。`Hello from Docker`のようなメッセージが表示されたらOK。

In [27]:
!ansible -b -a 'chdir=~/hello-compose /usr/local/bin/docker-compose up' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
Attaching to hello-compose_test-hello-world_1
test-hello-world_1  | 
test-hello-world_1  | Hello from Docker!
test-hello-world_1  | This message shows that your installation appears to be working correctly.
test-hello-world_1  | 
test-hello-world_1  | To generate this message, Docker took the following steps:
test-hello-world_1  |  1. The Docker client contacted the Docker daemon.
test-hello-world_1  |  2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
test-hello-world_1  |     (amd64)
test-hello-world_1  |  3. The Docker daemon created a new container from that image which runs the
test-hello-world_1  |     executable that produces the output you are currently reading.
test-hello-world_1  |  4. The Docker daemon streamed that output to the Docker client, which sent it
test-hello-world_1  |     to your terminal.
test-hello-world_1  | 
test-hello-world_1  | To try something more ambitious, you can run an Ubuntu container with:
test-h

# Docker Swarmの準備

Swarm Modeを有効化する。

https://docs.docker.com/engine/swarm/swarm-tutorial/create-swarm/

## 前提条件の確認

Docker Swarmは、Overlay Networkを使用するので、以下の前提条件が満たされていることを確認する。

https://docs.docker.com/network/overlay/ 

の、Create an overlay networkの、Prerequisitesが満たされているか確認する。

> AWSで構築している場合は問題なしのはず...

In [28]:
# 前提条件が満たされていることを確認して、Freezeして先に進める
assert False

AssertionError: 

## ノードのroleを設定

Swarmのmanager/worker(master/slave)と、Swarmの通信に使用するIPアドレスは、Inventoryから設定する。


masterとするAnsibleのgroup名を指定する。対象が正しいか、Ansible pingモジュールで動作確認する。

In [29]:
target_master = 'cwhub_master'

!ansible -m ping {target_master} -l {target_group}

10.1.1.236 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


workerとするnodeはAnsibleのgroup名で定義する。Ansible pingモジュールで動作確認する。

In [30]:
target_nodes = 'cwhub_nodes'

!ansible -m ping {target_nodes} -l {target_group}

10.1.1.157 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.144 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


各々のノードには、servicenet_ip変数が定義されている必要がある。

In [31]:
!ansible -m debug -a 'msg={{{{ servicenet_ip }}}}' {target_master} -l {target_group}
!ansible -m debug -a 'msg={{{{ servicenet_ip }}}}' {target_nodes} -l {target_group}

10.1.1.236 | SUCCESS => {
    "msg": "10.1.1.236"
}
10.1.1.144 | SUCCESS => {
    "msg": "10.1.1.144"
}
10.1.1.157 | SUCCESS => {
    "msg": "10.1.1.157"
}


## hostsの設定

`/etc/hosts`を設定する。

Swarm Nodeの通信に使うIPアドレスが、互いに名前が解決できるようにしておく。（hostsに登録されてないと名前解決できない環境のための設定）

In [32]:
!ansible -a hostname {target_group}

10.1.1.157 | CHANGED | rc=0 >>
ip-10-1-1-157.ap-northeast-1.compute.internal
10.1.1.144 | CHANGED | rc=0 >>
ip-10-1-1-144.ap-northeast-1.compute.internal
10.1.1.236 | CHANGED | rc=0 >>
ip-10-1-1-236.ap-northeast-1.compute.internal


inventory上のhostnameと設定されているhostnameの関係を調べる

In [33]:
import re
ping_pattern = re.compile(r'^(\S+)\s*\|\s*(?:SUCCESS|CHANGED)\s*.*\>\>.*$')
hostname_result = !ansible -a hostname {target_group}
hostname_result = [(ping_pattern.match(l1).group(1), l2) for l1, l2 in zip(hostname_result, hostname_result[1:]) if ping_pattern.match(l1)]
hostname_result

[('10.1.1.144', 'ip-10-1-1-144.ap-northeast-1.compute.internal'),
 ('10.1.1.157', 'ip-10-1-1-157.ap-northeast-1.compute.internal'),
 ('10.1.1.236', 'ip-10-1-1-236.ap-northeast-1.compute.internal')]

Docker Swarmの通信に使用するIPアドレス(VMのIPアドレス、インベントリの`servicenet_ip`)と、ホスト名の関係を調べる。

In [34]:
%%writefile {temp_dir}/service_ip2hostname.yml
---
- become: true
  hosts: all
  tasks:
      - debug: "msg={{ansible_hostname}}={{servicenet_ip}}"


Writing /tmp/tmpbee44q4h/service_ip2hostname.yml


In [35]:
import re

msgout = !ansible-playbook -l {target_group} {temp_dir}/service_ip2hostname.yml
msgout = msgout.grep('"msg"')
assert len(msgout) >= 1
hostname_result = []
for l in msgout:
    m = re.match(r'.*\"msg\": \"(.+)=(.+)\"', l)
    if m:
        hostname_result.append((m.group(2), m.group(1)))
hostname_result

[('10.1.1.236', 'ip-10-1-1-236'),
 ('10.1.1.144', 'ip-10-1-1-144'),
 ('10.1.1.157', 'ip-10-1-1-157')]

現在のhostsを確認しておく

In [36]:
!ansible -a 'cat /etc/hosts' {target_group}

10.1.1.157 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6
10.1.1.236 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6
10.1.1.144 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6


設定する

In [37]:
for ipaddr, hostname in hostname_result:
    !ansible -b -CDv -m lineinfile -a 'dest=/etc/hosts regexp="^{ipaddr} " line="{ipaddr} {hostname}"' {target_group}

Using /etc/ansible/ansible.cfg as config file
--- before: /etc/hosts (content)
+++ after: /etc/hosts (content)
@@ -1,3 +1,4 @@
 127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
 ::1         localhost localhost.localdomain localhost6 localhost6.localdomain6
 
+10.1.1.236 ip-10-1-1-236

10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
--- before: /etc/hosts (content)
+++ after: /etc/hosts (content)
@@ -1,3 +1,4 @@
 127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
 ::1         localhost localhost.localdomain localhost6 localhost6.localdomain6
 
+10.1.1.236 ip-10-1-1-236

10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
--- before: /etc/hosts (content)
+++ after: /etc/hosts

In [38]:
for ipaddr, hostname in hostname_result:
    !ansible -b -m lineinfile -a 'dest=/etc/hosts regexp="^{ipaddr} " line="{ipaddr} {hostname}"' {target_group}

10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
10.1.1.144 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
10.1.1.236 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "backup": "",
    "changed": true,
    "msg": "line added"
}
10.1.1.157 | CHANGED => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/

hostsを確認しておく

In [39]:
!ansible -a 'cat /etc/hosts' {target_group}

10.1.1.236 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6

10.1.1.236 ip-10-1-1-236
10.1.1.144 ip-10-1-1-144
10.1.1.157 ip-10-1-1-157
10.1.1.157 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6

10.1.1.236 ip-10-1-1-236
10.1.1.144 ip-10-1-1-144
10.1.1.157 ip-10-1-1-157
10.1.1.144 | CHANGED | rc=0 >>
127.0.0.1   localhost localhost.localdomain localhost4 localhost4.localdomain4
::1         localhost localhost.localdomain localhost6 localhost6.localdomain6

10.1.1.236 ip-10-1-1-236
10.1.1.144 ip-10-1-1-144
10.1.1.157 ip-10-1-1-157


## Setting up the Master

まず、managerノードとなるDocker EngineのIPアドレスを取得する。

In [40]:
master_ip_stdout = !ansible -m ping {target_master} -l {target_group}
manager_ip = [line.split()[0] for line in master_ip_stdout if 'SUCCESS' in line][0]
manager_ip

'10.1.1.236'

Docker Swarmを初期化する。

In [41]:
!ansible -b -a 'docker swarm init --advertise-addr {manager_ip}' {target_master} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
Swarm initialized: current node (k6eaebk7cjhgq11b2n03fawff) is now a manager.

To add a worker to this swarm, run the following command:

    docker swarm join --token SWMTKN-1-57i6jrx9f8tc1w9uacvemgjp2szv36eruvknv2mpo5r26mprjl-8bi5v54cmqlq7ncim3g8az0a8 10.1.1.236:2377

To add a manager to this swarm, run 'docker swarm join-token manager' and follow the instructions.


上記セルの実行結果に、Swarm Tokenが表示されるので、これを以下の変数に控えておく。(IPアドレスも合わせて)

In [42]:
# このセルを実行すると、テキストフィールドが現れるので、docker swarm join以降を入力する。
swarm_token = input()
if 'docker swarm join --token' in swarm_token.lstrip():
    swarm_token = swarm_token.lstrip()[len('docker swarm join --token'):].strip()
swarm_token

docker swarm join --token SWMTKN-1-57i6jrx9f8tc1w9uacvemgjp2szv36eruvknv2mpo5r26mprjl-8bi5v54cmqlq7ncim3g8az0a8 10.1.1.236:2377


'SWMTKN-1-57i6jrx9f8tc1w9uacvemgjp2szv36eruvknv2mpo5r26mprjl-8bi5v54cmqlq7ncim3g8az0a8 10.1.1.236:2377'

`docker info`で、`Swarm: active`となっていることを確認。

In [43]:
!ansible -b -a 'docker info' {target_master} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
Client:
 Context:    default
 Debug Mode: false
 Plugins:
  app: Docker App (Docker Inc., v0.9.1-beta3)
  buildx: Build with BuildKit (Docker Inc., v0.6.1-docker)
  scan: Docker Scan (Docker Inc., v0.8.0)

Server:
 Containers: 2
  Running: 0
  Paused: 0
  Stopped: 2
 Images: 1
 Server Version: 20.10.8
 Storage Driver: overlay2
  Backing Filesystem: xfs
  Supports d_type: true
  Native Overlay Diff: true
  userxattr: false
 Logging Driver: json-file
 Cgroup Driver: cgroupfs
 Cgroup Version: 1
 Plugins:
  Volume: local
  Network: bridge host ipvlan macvlan null overlay
  Log: awslogs fluentd gcplogs gelf journald json-file local logentries splunk syslog
 Swarm: active
  NodeID: k6eaebk7cjhgq11b2n03fawff
  Is Manager: true
  ClusterID: t386cromr0hu9z5k2rwpy9jt0
  Managers: 1
  Nodes: 1
  Default Address Pool: 10.0.0.0/8  
  SubnetSize: 24
  Data Path Port: 4789
  Orchestration:
   Task History Retention Limit: 5
  Raft:
   Snapshot Interval: 10000
   Number 

現在は1ノードで動作しているはず。

In [44]:
!ansible -b -a 'docker node ls' {target_master} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
ID                            HOSTNAME                                        STATUS    AVAILABILITY   MANAGER STATUS   ENGINE VERSION
k6eaebk7cjhgq11b2n03fawff *   ip-10-1-1-236.ap-northeast-1.compute.internal   Ready     Active         Leader           20.10.8


## Add nodes to the swarm

https://docs.docker.com/engine/swarm/swarm-tutorial/add-nodes/

In [45]:
!ansible -b -a 'docker swarm join --token {swarm_token}' {target_nodes} -l {target_group}

10.1.1.144 | CHANGED | rc=0 >>
This node joined a swarm as a worker.
10.1.1.157 | CHANGED | rc=0 >>
This node joined a swarm as a worker.


ノードは追加されたか？

In [46]:
!ansible -b -a 'docker node ls' {target_master} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
ID                            HOSTNAME                                        STATUS    AVAILABILITY   MANAGER STATUS   ENGINE VERSION
y3hjrr2lgkkr6iu98jxqvh1ne     ip-10-1-1-144.ap-northeast-1.compute.internal   Ready     Active                          20.10.8
w74m819x2b9q4qstbm4t65y23     ip-10-1-1-157.ap-northeast-1.compute.internal   Ready     Active                          20.10.8
k6eaebk7cjhgq11b2n03fawff *   ip-10-1-1-236.ap-northeast-1.compute.internal   Ready     Active         Leader           20.10.8


これでSwarmが構成されたはず

# 後始末

In [47]:
!rm -rf {temp_dir}